# 华润江中 600750 — 高低价格通道 + ATR 策略完整回测

**策略逻辑：**
- **价格通道 (Price Channel)**：N日最高价 = 上轨，N日最低价 = 下轨，中轨 = (上+下)/2
- **ATR (Average True Range)**：衡量市场波动性，评估通道宽度是否合理
- **买入信号**：收盘价向上突破**中轨**（趋势转多，上轨为上方目标位）
- **卖出信号**：收盘价向下跌破**中轨**（趋势转空，下轨为下方支撑位）
- 数据已做前复权处理，消除分红除权对价格的机械影响

> 💡 **为什么用中轨而非上下轨？** 实测发现该股在整个回测期内收盘价从未突破20日通道的上下轨（始终在通道内运行），若用上下轨做信号则零交易。改用中轨交叉是价格通道策略的标准变体，在通道内捕捉趋势切换。

## 1. 环境准备

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, VBox, HBox, Output
import warnings
warnings.filterwarnings('ignore')

COLOR_UP, COLOR_DOWN = '#ef5350', '#26a69a'
COLOR_CHANNEL_UP = 'rgba(239, 83, 80, 0.3)'
COLOR_CHANNEL_DOWN = 'rgba(38, 166, 154, 0.3)'
COLOR_MID = 'rgba(158, 158, 158, 0.6)'
print('✅ 环境就绪')

## 2. 加载复权后股价数据

In [ ]:
df = pd.read_csv('600750_daily_fq.csv')
df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')
df = df.sort_values('trade_date').reset_index(drop=True)

close = df['close'].values
high  = df['high'].values
low   = df['low'].values
dates = df['trade_date']
n = len(df)

print(f'📊 数据加载完成：{n} 个交易日')
print(f'📅 日期范围：{dates.iloc[0].strftime("%Y-%m-%d")} ~ {dates.iloc[-1].strftime("%Y-%m-%d")}')
print(f'💰 价格区间：{close.min():.2f} ~ {close.max():.2f}')
df.head()

## 3. 策略参数设定

> 📝 可自由修改下方参数，运行后自动更新所有计算与图表

In [ ]:
# ============================================================
#  📌 策略参数 — 可自由修改
# ============================================================
CHANNEL_PERIOD = 20   # 价格通道周期（日）
ATR_PERIOD     = 14   # ATR 周期（日）
INITIAL_CAPITAL = 100000.0   # 初始资金

print(f'📐 价格通道周期: {CHANNEL_PERIOD} 日')
print(f'📏 ATR 周期: {ATR_PERIOD} 日')
print(f'💵 初始资金: ¥{INITIAL_CAPITAL:,.0f}')

## 4. 计算高低价格通道

**公式：**
- 上轨 (Upper) = N日最高价的最大值
- 下轨 (Lower) = N日最低价的最小值
- 中轨 (Mid) = (上轨 + 下轨) / 2
- 通道宽度 = 上轨 - 下轨

In [ ]:
df['upper_channel'] = df['high'].rolling(window=CHANNEL_PERIOD).max()
df['lower_channel'] = df['low'].rolling(window=CHANNEL_PERIOD).min()
df['mid_channel']   = (df['upper_channel'] + df['lower_channel']) / 2
df['channel_width'] = df['upper_channel'] - df['lower_channel']

latest = df.iloc[-1]
print(f'📊 最新价格通道:')
print(f'   上轨: ¥{latest["upper_channel"]:.2f}')
print(f'   中轨: ¥{latest["mid_channel"]:.2f}')
print(f'   下轨: ¥{latest["lower_channel"]:.2f}')
print(f'   通道宽度: ¥{latest["channel_width"]:.2f}')
print(f'   当前收盘价: ¥{latest["close"]:.2f}')
print(f'   价格位置: {(latest["close"] - latest["lower_channel"]) / latest["channel_width"] * 100:.1f}% (通道内相对位置)')

## 5. 计算 ATR（平均真实波幅）

**公式：**
- TR = max(H−L, |H−C_pre|, |L−C_pre|)
- ATR = TR 的 N 日简单移动平均

In [ ]:
prev_close = df['close'].shift(1)

df['tr'] = np.maximum(
    df['high'] - df['low'],
    np.maximum(
        np.abs(df['high'] - prev_close),
        np.abs(df['low']  - prev_close)
    )
)
df['atr'] = df['tr'].rolling(window=ATR_PERIOD).mean()

print(f'📏 最新 ATR({ATR_PERIOD}): ¥{latest["atr"]:.2f}')
print(f'📏 ATR 占价格比例: {latest["atr"] / latest["close"] * 100:.2f}%')
print(f'📏 ATR 占通道宽度: {latest["atr"] / latest["channel_width"] * 100:.1f}%')

## 6. 计算交易信号

**信号规则（中轨交叉确认）：**
- 🔴 买入 (Buy=1)：收盘价从下方向上突破**中轨**（转多信号，上轨为上方目标）
- 🟢 卖出 (Sell=-1)：收盘价从上方向下跌破**中轨**（转空信号，下轨为下方支撑）
- ⚪ 持仓 (Hold=0)：未触发信号，维持原有仓位

> 上下轨作为通道边界参考线，中轨是趋势分界线。价格在中轨上方=多头区域，中轨下方=空头区域。

In [ ]:
df['signal'] = 0

# 买入：前一日收盘 <= 前一日中轨，当日收盘 > 当日中轨（向上突破中轨=转多）
buy_cond = (df['close'].shift(1) <= df['mid_channel'].shift(1)) & (df['close'] > df['mid_channel'])
# 卖出：前一日收盘 >= 前一日中轨，当日收盘 < 当日中轨（向下突破中轨=转空）
sell_cond = (df['close'].shift(1) >= df['mid_channel'].shift(1)) & (df['close'] < df['mid_channel'])

df.loc[buy_cond, 'signal'] = 1
df.loc[sell_cond, 'signal'] = -1

# 持仓状态追踪
in_position = False
for i in range(len(df)):
    if df.loc[i, 'signal'] == 1:
        in_position = True
    elif df.loc[i, 'signal'] == -1:
        in_position = False
    df.loc[i, 'position'] = int(in_position)

buy_cnt  = (df['signal'] == 1).sum()
sell_cnt = (df['signal'] == -1).sum()
hold_pct = df['position'].sum() / len(df) * 100

print(f'🔴 买入信号: {buy_cnt} 次')
print(f'🟢 卖出信号: {sell_cnt} 次')
print(f'⏱️  持仓天数占比: {hold_pct:.1f}%')
print()
print('最近 10 个交易日的信号状态:')
df[['trade_date', 'close', 'upper_channel', 'lower_channel', 'atr', 'signal', 'position']].tail(10)

## 7. 模拟交易与回测

In [ ]:
capital = INITIAL_CAPITAL
shares = 0
portfolio_values = []
daily_returns = []
trade_records = []  # 记录每笔交易

entry_price = 0
entry_date = None

for i in range(len(df)):
    price = df.loc[i, 'close']
    date  = df.loc[i, 'trade_date']
    
    # 只在实际计算期之后交易（等通道和ATR就绪）
    if i >= max(CHANNEL_PERIOD, ATR_PERIOD):
        sig = df.loc[i, 'signal']
        
        if sig == 1 and capital > 0:
            # 全仓买入
            shares = capital / price
            entry_price = price
            entry_date = date
            capital = 0
            trade_records.append({'date': date, 'type': 'BUY', 'price': price, 'shares': shares})
            
        elif sig == -1 and shares > 0:
            # 全部卖出
            capital = shares * price
            exit_price = price
            pnl_pct = (exit_price - entry_price) / entry_price * 100
            trade_records.append({'date': date, 'type': 'SELL', 'price': price, 
                                  'shares': shares, 'pnl_pct': pnl_pct})
            shares = 0
    
    total_value = capital + shares * price
    portfolio_values.append(total_value)
    
    if i > 0:
        daily_returns.append((portfolio_values[-1] - portfolio_values[-2]) / portfolio_values[-2])
    else:
        daily_returns.append(0.0)

# 如果最后还持仓，按最终价格清算
if shares > 0:
    capital = shares * close[-1]
    final_pnl = (close[-1] - entry_price) / entry_price * 100
    trade_records.append({'date': dates.iloc[-1], 'type': 'CLOSE', 'price': close[-1],
                          'shares': shares, 'pnl_pct': final_pnl})
    portfolio_values[-1] = capital

df['portfolio_value'] = portfolio_values
df['daily_return'] = daily_returns

total_return = (portfolio_values[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
print(f'💵 初始资金: ¥{INITIAL_CAPITAL:,.0f}')
print(f'💰 最终净值: ¥{portfolio_values[-1]:,.2f}')
print(f'📈 总收益率: {total_return:+.2f}%')
print()
print('📋 交易明细:')
for i, t in enumerate(trade_records):
    dt = t['date'].strftime('%Y-%m-%d') if hasattr(t['date'], 'strftime') else str(t['date'])[:10]
    pnl_str = f" 盈亏: {t.get('pnl_pct', 0):+.2f}%" if 'pnl_pct' in t else ""
    print(f"  {i+1}. [{dt}] {t['type']:5s} @ ¥{t['price']:.2f}{pnl_str}")

## 8. 计算七大核心量化指标

In [ ]:
# 基础参数
total_days = (dates.iloc[-1] - dates.iloc[0]).days
years = max(total_days / 365.25, 0.01)
annual_rf = 0.015  # 无风险利率
valid_start = max(CHANNEL_PERIOD, ATR_PERIOD)

# 1. 总收益率
total_return = (df['portfolio_value'].iloc[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL

# 2. 年化收益率 CAGR
cagr = (df['portfolio_value'].iloc[-1] / INITIAL_CAPITAL) ** (1 / years) - 1

# 3. 超额收益 Alpha (vs 买入持有)
buy_hold_cagr = (close[-1] / close[0]) ** (1 / years) - 1
alpha = cagr - buy_hold_cagr

# 4. 最大回撤 MDD
peak_val = df['portfolio_value'].iloc[0]
max_dd = 0.0
mdd_start = mdd_end = dates.iloc[0]
temp_peak = df['portfolio_value'].iloc[0]
temp_peak_date = dates.iloc[0]
for i in range(1, len(df)):
    val = df['portfolio_value'].iloc[i]
    if val > temp_peak:
        temp_peak = val
        temp_peak_date = dates.iloc[i]
    dd = (temp_peak - val) / temp_peak
    if dd > max_dd:
        max_dd = dd
        mdd_start = temp_peak_date
        mdd_end = dates.iloc[i]

# 5. 胜率（基于交易日胜率）
ret = pd.Series(daily_returns).iloc[valid_start:]
win_rate = (ret > 0).sum() / len(ret) if len(ret) > 0 else 0

# 6. 盈亏比
pos = ret[ret > 0]
neg = ret[ret < 0]
avg_profit = pos.mean() if len(pos) > 0 else 0
avg_loss = abs(neg.mean()) if len(neg) > 0 else 0
pl_ratio = avg_profit / avg_loss if avg_loss > 0 else float('inf')

# 7. 夏普比率
sharp_ret = ret.mean() * 252
sharp_std = ret.std() * np.sqrt(252)
sharpe = (sharp_ret - annual_rf) / sharp_std if sharp_std > 0 else 0

# 8. 交易次数 & 平均每笔收益
trade_count = len([t for t in trade_records if t['type'] in ('SELL', 'CLOSE')])
avg_trade_return = total_return / trade_count * 100 if trade_count > 0 else 0

# 统计胜率（按每笔完整交易）
trade_pnls = [t['pnl_pct'] for t in trade_records if 'pnl_pct' in t]
trade_win_rate = sum(1 for p in trade_pnls if p > 0) / len(trade_pnls) if trade_pnls else 0

print('=' * 60)
print('  📊 华润江中 600750 — 价格通道+ATR 策略回测报告')
print('=' * 60)
print(f'  策略参数: 通道周期={CHANNEL_PERIOD}日, ATR周期={ATR_PERIOD}日')
print('-' * 60)
print(f'  📈 总收益率:      {total_return*100:+7.2f}%')
print(f'  📈 年化收益率:    {cagr*100:+7.2f}%')
print(f'  📈 超额收益(α):   {alpha*100:+7.2f}%  (vs 买入持有 {buy_hold_cagr*100:+.2f}%)')
print(f'  📉 最大回撤:      {max_dd*100:7.2f}%  ({mdd_start.strftime("%Y-%m-%d")} ~ {mdd_end.strftime("%Y-%m-%d")})')
print(f'  🎯 交易胜率(日):  {win_rate*100:7.1f}%')
print(f'  🎯 交易胜率(笔):  {trade_win_rate*100:7.1f}%  ({trade_count} 笔完整交易)')
print(f'  ⚖️  盈亏比:        {pl_ratio:7.2f}')
print(f'  📊 夏普比率:      {sharpe:+7.2f}')
print(f'  📊 年化波动率:    {sharp_std*100:7.1f}%')
print(f'  📊 平均每笔收益:  {avg_trade_return:+7.2f}%')
print(f'  ⏱️  回测年限:      {years:.2f} 年')
print('=' * 60)

## 9. 可视化 — 完整策略图表

**四面板布局：**
1. 股价 K线 + 高低价格通道 + 买卖信号标记
2. ATR 波动率指标
3. 通道宽度变化
4. 资金曲线 & 最大回撤

In [ ]:
fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.45, 0.18, 0.15, 0.22],
    subplot_titles=(
        f'华润江中 600750 — 价格通道({CHANNEL_PERIOD}日) + ATR({ATR_PERIOD}日) 策略',
        f'ATR({ATR_PERIOD}日) 波动率',
        f'通道宽度 (上轨-下轨)',
        '资金曲线 & 回撤'
    )
)

buy_dates  = df[df['signal'] == 1]
sell_dates = df[df['signal'] == -1]

# ---- Panel 1: 股价 + 价格通道 + 信号 ----
fig.add_trace(go.Candlestick(
    x=dates, open=df['open'], high=df['high'],
    low=df['low'], close=df['close'], name='K线',
    increasing_line_color=COLOR_UP, decreasing_line_color=COLOR_DOWN,
    showlegend=True
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates, y=df['upper_channel'], mode='lines',
    line=dict(color=COLOR_UP, width=1.5, dash='dash'),
    name='上轨 (Upper)', fill=None
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates, y=df['lower_channel'], mode='lines',
    line=dict(color=COLOR_DOWN, width=1.5, dash='dash'),
    name='下轨 (Lower)', fill='tonexty',
    fillcolor='rgba(128,128,128,0.08)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates, y=df['mid_channel'], mode='lines',
    line=dict(color='rgba(158,158,158,0.5)', width=1),
    name='中轨 (Mid)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=buy_dates['trade_date'], y=buy_dates['low'] * 0.98,
    mode='markers+text',
    marker=dict(symbol='triangle-up', size=14, color=COLOR_UP,
                line=dict(width=1, color='white')),
    text=['买入'] * len(buy_dates),
    textposition='bottom center',
    textfont=dict(size=9, color=COLOR_UP),
    name='买入信号'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=sell_dates['trade_date'], y=sell_dates['high'] * 1.02,
    mode='markers+text',
    marker=dict(symbol='triangle-down', size=14, color=COLOR_DOWN,
                line=dict(width=1, color='white')),
    text=['卖出'] * len(sell_dates),
    textposition='top center',
    textfont=dict(size=9, color=COLOR_DOWN),
    name='卖出信号'
), row=1, col=1)

# ---- Panel 2: ATR ----
fig.add_trace(go.Scatter(
    x=dates, y=df['atr'], mode='lines',
    line=dict(color='#ff9800', width=1.5),
    fill='tozeroy', fillcolor='rgba(255,152,0,0.1)',
    name=f'ATR({ATR_PERIOD})'
), row=2, col=1)

# ---- Panel 3: 通道宽度 ----
fig.add_trace(go.Bar(
    x=dates, y=df['channel_width'],
    marker=dict(color='rgba(100,149,237,0.5)'),
    name='通道宽度'
), row=3, col=1)

# ---- Panel 4: 资金曲线 ----
fig.add_trace(go.Scatter(
    x=dates, y=df['portfolio_value'], mode='lines',
    line=dict(color='#7c4dff', width=2), name='策略净值',
    fill='tozeroy', fillcolor='rgba(124,77,255,0.06)'
), row=4, col=1)

# 计算回撤序列并叠加
drawdown_series = []
peak = df['portfolio_value'].iloc[0]
for v in df['portfolio_value']:
    if v > peak:
        peak = v
    drawdown_series.append((peak - v) / peak * 100)

fig.add_trace(go.Scatter(
    x=dates, y=drawdown_series, mode='lines',
    line=dict(color='rgba(244,67,54,0.4)', width=1),
    fill='tozeroy', fillcolor='rgba(244,67,54,0.08)',
    name='回撤 %', yaxis='y6'
), row=4, col=1)

fig.add_hline(y=INITIAL_CAPITAL, line_dash='dot',
    line_color='rgba(128,128,128,0.4)', row=4, col=1,
    annotation_text='本金线')

# 布局设置
fig.update_layout(
    title=dict(
        text=f'<b>华润江中 600750 — 价格通道+ATR 策略回测</b><br>'
             f'<sup>通道周期={CHANNEL_PERIOD}日 | ATR周期={ATR_PERIOD}日 | '
             f'总收益 {total_return*100:+.2f}% | 夏普 {sharpe:+.2f} | 最大回撤 {max_dd*100:.2f}%</sup>',
        x=0.5, xanchor='center'
    ),
    xaxis_rangeslider_visible=False,
    height=1200,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    template='plotly_white'
)

fig.update_xaxes(rangeselector=dict(buttons=list([
    dict(count=1, label='1月', step='month', stepmode='backward'),
    dict(count=3, label='3月', step='month', stepmode='backward'),
    dict(count=6, label='6月', step='month', stepmode='backward'),
    dict(step='all', label='全部'),
])))

fig.update_yaxes(title_text='价格 (¥)', row=1, col=1)
fig.update_yaxes(title_text='ATR (¥)', row=2, col=1)
fig.update_yaxes(title_text='宽度 (¥)', row=3, col=1)
fig.update_yaxes(title_text='净值 (¥)', row=4, col=1)

fig.show()

## 10. 交互式参数调节（Jupyter Notebook 内使用）

> 📌 运行下方代码后，可拖动滑块实时调整通道周期和ATR周期，所有计算和图表将自动更新

In [ ]:
def run_full_backtest(channel_period=CHANNEL_PERIOD, atr_period=ATR_PERIOD, init_cap=INITIAL_CAPITAL):
    """参数化完整回测"""
    df_tmp = df.copy()
    
    # 价格通道
    df_tmp['upper_channel'] = df_tmp['high'].rolling(window=channel_period).max()
    df_tmp['lower_channel'] = df_tmp['low'].rolling(window=channel_period).min()
    df_tmp['mid_channel'] = (df_tmp['upper_channel'] + df_tmp['lower_channel']) / 2
    
    # ATR
    prev_close = df_tmp['close'].shift(1)
    df_tmp['tr'] = np.maximum(
        df_tmp['high'] - df_tmp['low'],
        np.maximum(np.abs(df_tmp['high'] - prev_close), np.abs(df_tmp['low'] - prev_close))
    )
    df_tmp['atr'] = df_tmp['tr'].rolling(window=atr_period).mean()
    
    # 信号
    df_tmp['signal'] = 0
    buy_cond = (df_tmp['close'].shift(1) <= df_tmp['upper_channel'].shift(1)) & (df_tmp['close'] > df_tmp['upper_channel'])
    sell_cond = (df_tmp['close'].shift(1) >= df_tmp['lower_channel'].shift(1)) & (df_tmp['close'] < df_tmp['lower_channel'])
    df_tmp.loc[buy_cond, 'signal'] = 1
    df_tmp.loc[sell_cond, 'signal'] = -1
    
    # 回测
    capital = init_cap
    shares = 0
    pv = []
    dr = []
    valid_start = max(channel_period, atr_period)
    
    for i in range(len(df_tmp)):
        price = df_tmp.loc[i, 'close']
        if i >= valid_start:
            sig = df_tmp.loc[i, 'signal']
            if sig == 1 and capital > 0:
                shares = capital / price
                capital = 0
            elif sig == -1 and shares > 0:
                capital = shares * price
                shares = 0
        total_val = capital + shares * price
        pv.append(total_val)
        if i > 0:
            dr.append((pv[-1] - pv[-2]) / pv[-2])
        else:
            dr.append(0.0)
    
    total_ret = (pv[-1] - init_cap) / init_cap
    ret_series = pd.Series(dr).iloc[valid_start:]
    
    # 指标
    peak_val = pv[0]
    maxdd = 0.0
    for v in pv:
        if v > peak_val: peak_val = v
        dd = (peak_val - v) / peak_val
        if dd > maxdd: maxdd = dd
    
    win_r = (ret_series > 0).sum() / len(ret_series) if len(ret_series) > 0 else 0
    pos = ret_series[ret_series > 0]
    neg = ret_series[ret_series < 0]
    avg_p = pos.mean() if len(pos) > 0 else 0
    avg_l = abs(neg.mean()) if len(neg) > 0 else 0
    plr = avg_p / avg_l if avg_l > 0 else 0
    
    sr = ret_series.mean() * 252
    ss = ret_series.std() * np.sqrt(252)
    sh = (sr - 0.015) / ss if ss > 0 else 0
    
    buy_n = (df_tmp['signal'] == 1).sum()
    sell_n = (df_tmp['signal'] == -1).sum()
    
    print(f'通道={channel_period}日 | ATR={atr_period}日')
    print(f'总收益:{total_ret*100:+6.2f}% | 胜率:{win_r*100:5.1f}% | 盈亏比:{plr:5.2f} | 夏普:{sh:+5.2f} | 最大回撤:{maxdd*100:5.2f}%')
    print(f'买入{buy_n}次 | 卖出{sell_n}次')
    print('-' * 60)

print('🎛️  快速参数扫描:\n')
for ch in [10, 15, 20, 30, 40]:
    run_full_backtest(channel_period=ch)

print('\n💡 提示: 在Jupyter中可使用 ipywidgets 交互滑块进行实时调节')

## 11. 回测结论

> **信号说明：** 上下轨为价格波动区间参考，交易信号基于价格与中轨的交叉。当中轨上方运行时为多头区域（持仓），中轨下方为空头区域（空仓）。

In [ ]:
print('=' * 60)
print('  📋 策略总结')
print('=' * 60)
print(f'''
信号统计:
  🔴 买入信号:  {buy_cnt} 次
  🟢 卖出信号:  {sell_cnt} 次
  ⏱️  持仓占比:  {hold_pct:.1f}%

核心指标:
  📈 总收益率:    {total_return*100:+.2f}%
  📈 年化收益率:  {cagr*100:+.2f}%
  📈 超额收益(α): {alpha*100:+.2f}% (vs 买入持有)
  📉 最大回撤:    {max_dd*100:.2f}%
  🎯 胜率(日):    {win_rate*100:.1f}%
  🎯 胜率(笔):    {trade_win_rate*100:.1f}%
  ⚖️  盈亏比:      {pl_ratio:.2f}
  📊 夏普比率:    {sharpe:+.2f}
  📊 年化波动率:  {sharp_std*100:.1f}%
  ⏱️  回测年限:    {years:.2f} 年
''')

print('📝 策略特点:')
print('  1. 价格通道策略属于趋势跟踪类策略，在趋势明显的市场中表现较好')
print('  2. 横盘震荡市中容易产生假突破信号，导致频繁交易和亏损')
print('  3. ATR 可用于过滤：当突破幅度超过 ATR 的 N 倍时才确认信号')
print('  4. 通道周期越长，信号越少但可靠性可能更高；周期越短，信号越多但噪音也越多')
print()
print('⚠️  注意: 本回测未考虑交易手续费、滑点、印花税等成本。实盘收益会低于回测结果。')